# IFN680 Project 4 — Case 1: Clean-Shaven Classifier Audit

## 1. Audit Objective

### Case context
A clean-shaven facial-image classifier achieved near-perfect performance during development on studio-quality data, but its accuracy decreased substantially after deployment in a mobile application.

### Audit question
**What changed between the internal ("Studio") test environment and the external ("Mobile App") environment, and can that change explain the observed performance degradation?**

### Initial hypothesis
The deployment failure may be caused by **distribution shift** between the internal and external image sets. Candidate differences include image resolution, compression, blur, noise, lighting, contrast, colour, cropping, background, and face scale.

This is an **initial audit hypothesis only**. The diagnosis will not be accepted until it is supported by experimental evidence.

### Audit workflow
1. Load the supplied model and both test datasets.
2. Inspect the datasets before evaluating the model.
3. Establish baseline internal and external performance.
4. Identify measurable differences between the two data distributions.
5. Test whether those differences are associated with model errors.
6. Use explainability techniques where useful to examine model behaviour.
7. Form a root-cause diagnosis and evidence-linked recommendations.

> **Assessment principle:** the goal is not merely to show that external accuracy is lower. The goal is to identify *why* it is lower and provide reproducible evidence supporting that conclusion.


## 2. Environment / Imports

The starter notebook uses PyTorch / torchvision, NumPy, pandas, Matplotlib and scikit-learn. The same core environment is retained here.

The random seeds are fixed for reproducibility. The notebook will use a CUDA GPU when one is available and otherwise fall back to CPU.

Because this notebook is stored in the project root, all Case 1 source files are referenced through the `Case1/` directory.


In [ ]:
from pathlib import Path
from collections import Counter
import random
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Progress bar used by the supplied starter notebook and later evaluation cells.
import tqdm

# -------------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------------
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# -------------------------------------------------------------------------
# Device
# -------------------------------------------------------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------------------------------------------------
# Project paths
# Expected structure:
#
# project4_code/
# ├── case1.ipynb
# └── Case1/
#     ├── Case1Dataset-2.zip
#     ├── student_start_notebook-4.ipynb
#     └── resnet_frozen_best-2.pth
# -------------------------------------------------------------------------
ROOT_DIR = Path.cwd()
CASE_DIR = ROOT_DIR / "Case1"

DATA_ZIP_PATH = CASE_DIR / "Case1Dataset-2.zip"
MODEL_PATH = CASE_DIR / "resnet_frozen_best-2.pth"

# Extracted data are kept inside Case1 so that rerunning the notebook is simple.
EXTRACT_DIR = CASE_DIR / "_extracted"

assert CASE_DIR.exists(), f"Case directory not found: {CASE_DIR}"
assert DATA_ZIP_PATH.exists(), f"Dataset ZIP not found: {DATA_ZIP_PATH}"
assert MODEL_PATH.exists(), f"Model weights not found: {MODEL_PATH}"

print(f"Case directory: {CASE_DIR}")
print(f"Dataset ZIP:    {DATA_ZIP_PATH}")
print(f"Model weights:  {MODEL_PATH}")


## 3. Load Model

The supplied starter notebook defines a **ResNet-18** classifier with a two-output fully connected layer and loads the provided trained state dictionary.

The architecture is retained. `weights=None` is used when constructing the backbone because the supplied `.pth` file contains the trained weights that are loaded immediately afterwards; this also avoids requiring an ImageNet download when the notebook is run in the IFN680 environment.


In [ ]:
def setup_model(model, num_classes, freeze_backbone=False):
    """Adapt a torchvision ResNet classifier to the required number of classes."""
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

        # Keep the final classification layer trainable.
        for param in model.fc.parameters():
            param.requires_grad = True

    return model


# Recreate the architecture used by the supplied starter notebook.
backbone = torchvision.models.resnet18(weights=None)
resnet_frozen = setup_model(backbone, num_classes=2, freeze_backbone=False)

# Load the supplied trained weights.
# weights_only=True is preferred when supported by the installed PyTorch version.
try:
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    checkpoint = torch.load(MODEL_PATH, map_location=device)

# Support either a raw state_dict or a checkpoint containing a "state_dict" entry.
state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint

resnet_frozen.load_state_dict(state_dict)
resnet_frozen = resnet_frozen.to(device)
resnet_frozen.eval()

print("Model loaded successfully.")
print(f"Model type: {resnet_frozen.__class__.__name__}")
print(f"Output classes: {resnet_frozen.fc.out_features}")


## 4. Load Internal Dataset

The internal test set represents the development / "Studio" distribution.

The supplied starter notebook uses `torchvision.datasets.ImageFolder`, so class labels are inferred directly from the dataset's class-folder names. We retain those labels rather than hard-coding assumptions about which numerical class represents "clean shaven".

The model uses the same ImageNet normalisation as the starter notebook:

- mean = `(0.485, 0.456, 0.406)`
- standard deviation = `(0.229, 0.224, 0.225)`
- input size = `224 × 224`

A second **raw** `ImageFolder` is also kept for visual and image-property inspection without normalisation.


In [ ]:
# -------------------------------------------------------------------------
# Extract the supplied dataset ZIP
# -------------------------------------------------------------------------
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Only extract when the working directory is empty.
# This keeps repeated notebook runs fast and avoids unnecessary writes.
if not any(EXTRACT_DIR.iterdir()):
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f"Dataset extracted to: {EXTRACT_DIR}")
else:
    print(f"Using existing extracted dataset at: {EXTRACT_DIR}")


def find_unique_split(root: Path, split_name: str) -> Path:
    """Find one ImageFolder split directory beneath an extracted dataset root."""
    matches = [p for p in root.rglob(split_name) if p.is_dir()]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find a directory named '{split_name}' beneath {root}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Found multiple directories named '{split_name}': {matches}"
        )

    return matches[0]


INTERNAL_DIR = find_unique_split(EXTRACT_DIR, "test_internal")
print(f"Internal split: {INTERNAL_DIR}")

# -------------------------------------------------------------------------
# Model preprocessing from the supplied starter notebook
# -------------------------------------------------------------------------
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(imagenet_means, imagenet_stds),
])

# Keep a raw version for audit inspection and a transformed version for inference.
test_internal_raw = torchvision.datasets.ImageFolder(INTERNAL_DIR)
test_internal_dataset = torchvision.datasets.ImageFolder(
    INTERNAL_DIR,
    transform=transform
)

batch_size = 16
test_internal_loader = torch.utils.data.DataLoader(
    test_internal_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"Internal test dataset size: {len(test_internal_dataset)}")
print(f"Internal classes: {test_internal_dataset.classes}")
print(f"Internal class mapping: {test_internal_dataset.class_to_idx}")


## 5. Load External Dataset

The external test set represents the field / "Mobile App" distribution where the deployed model is reported to fail.

It is loaded with exactly the same preprocessing as the internal set so that subsequent performance comparisons reflect the supplied model pipeline rather than differences introduced by the audit code.


In [ ]:
EXTERNAL_DIR = find_unique_split(EXTRACT_DIR, "test_external")
print(f"External split: {EXTERNAL_DIR}")

# Raw copy for visual / image-property inspection.
test_external_raw = torchvision.datasets.ImageFolder(EXTERNAL_DIR)

# Transformed copy for model inference.
test_external_dataset = torchvision.datasets.ImageFolder(
    EXTERNAL_DIR,
    transform=transform
)

test_external_loader = torch.utils.data.DataLoader(
    test_external_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"External test dataset size: {len(test_external_dataset)}")
print(f"External classes: {test_external_dataset.classes}")
print(f"External class mapping: {test_external_dataset.class_to_idx}")

# A valid direct comparison requires the same class-index mapping in both sets.
assert test_internal_dataset.class_to_idx == test_external_dataset.class_to_idx, (
    "Internal and external class mappings differ. "
    "Do not compare model outputs until this is resolved."
)

class_names = test_internal_dataset.classes
class_to_idx = test_internal_dataset.class_to_idx

print("\nClass mappings are consistent across both test sets.")


## 6. Dataset Inspection

Before evaluating the model, inspect whether the internal and external test sets are comparable.

This section establishes four basic properties:

1. **Dataset size**
2. **Class labels and class balance**
3. **Native image dimensions / resolution**
4. **Representative raw images from each class and split**

These checks are descriptive. They may suggest later hypotheses, but a visible difference alone is **not yet evidence that the difference causes the model failure**. Any suspected distribution shift must later be tested against model performance and errors.


### 6.1 Dataset size, labels and class balance

A major class-balance difference could change aggregate accuracy even when per-class behaviour is similar. We therefore compare both counts and proportions before interpreting later performance metrics.


In [ ]:
def class_distribution(dataset, split_name):
    counts = Counter(dataset.targets)
    total = len(dataset)

    rows = []
    for class_idx, class_name in enumerate(dataset.classes):
        count = counts.get(class_idx, 0)
        rows.append({
            "split": split_name,
            "class_index": class_idx,
            "class_name": class_name,
            "count": count,
            "proportion": count / total if total else np.nan,
        })

    return pd.DataFrame(rows)


class_balance = pd.concat([
    class_distribution(test_internal_raw, "internal"),
    class_distribution(test_external_raw, "external"),
], ignore_index=True)

display(class_balance)

# Side-by-side count comparison.
pivot_counts = class_balance.pivot(
    index="class_name",
    columns="split",
    values="count"
)

ax = pivot_counts.plot(kind="bar", figsize=(8, 5))
ax.set_title("Class Distribution: Internal vs External Test Sets")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


### 6.2 Native image dimensions

The model ultimately receives every image resized to `224 × 224`, but the **native dimensions** can still reveal a source-domain difference between studio and mobile data.

At this stage we inspect width, height and aspect ratio only. More targeted image-quality measures such as blur, brightness, contrast, compression artefacts or noise can be added later if the baseline results and visual inspection justify those hypotheses.


In [ ]:
def collect_image_metadata(dataset, split_name):
    records = []

    # ImageFolder.samples contains (file_path, class_index).
    for image_path, class_idx in tqdm.tqdm(
        dataset.samples,
        desc=f"Inspecting {split_name} image metadata"
    ):
        with Image.open(image_path) as img:
            width, height = img.size

        records.append({
            "split": split_name,
            "class_index": class_idx,
            "class_name": dataset.classes[class_idx],
            "width": width,
            "height": height,
            "aspect_ratio": width / height if height else np.nan,
            "pixels": width * height,
        })

    return pd.DataFrame(records)


image_metadata = pd.concat([
    collect_image_metadata(test_internal_raw, "internal"),
    collect_image_metadata(test_external_raw, "external"),
], ignore_index=True)

dimension_summary = (
    image_metadata
    .groupby("split")[["width", "height", "aspect_ratio", "pixels"]]
    .agg(["min", "median", "mean", "max"])
    .round(2)
)

display(dimension_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for split in ["internal", "external"]:
    subset = image_metadata[image_metadata["split"] == split]
    axes[0].hist(subset["width"], bins=20, alpha=0.6, label=split)
    axes[1].hist(subset["height"], bins=20, alpha=0.6, label=split)

axes[0].set_title("Native Image Width")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Number of images")
axes[0].legend()

axes[1].set_title("Native Image Height")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Number of images")
axes[1].legend()

plt.tight_layout()
plt.show()


### 6.3 Representative raw images

The following samples are shown **before resizing and ImageNet normalisation**. Sampling is stratified by class so that both labels are represented in both splits.

When inspecting the images, look for systematic differences between the internal and external domains rather than isolated unusual examples. Relevant observations may include:

- sharpness / blur;
- illumination;
- contrast and colour;
- image noise or compression;
- crop and face scale;
- pose;
- background;
- occlusion;
- any feature that appears correlated with the target class.

Any candidate difference identified here should become a testable hypothesis in the later case-specific investigation.


In [ ]:
def stratified_sample_paths(dataset, n_per_class=3, seed=SEED):
    rng = random.Random(seed)
    sampled = []

    for class_idx, class_name in enumerate(dataset.classes):
        candidates = [
            path for path, target in dataset.samples
            if target == class_idx
        ]

        n = min(n_per_class, len(candidates))
        for image_path in rng.sample(candidates, n):
            sampled.append((image_path, class_idx, class_name))

    return sampled


def show_representative_samples(dataset, split_name, n_per_class=3, seed=SEED):
    samples = stratified_sample_paths(
        dataset,
        n_per_class=n_per_class,
        seed=seed
    )

    n_classes = len(dataset.classes)
    n_cols = n_per_class
    fig, axes = plt.subplots(
        n_classes,
        n_cols,
        figsize=(4 * n_cols, 4 * n_classes),
        squeeze=False
    )

    for class_idx, class_name in enumerate(dataset.classes):
        class_samples = [s for s in samples if s[1] == class_idx]

        for col in range(n_cols):
            ax = axes[class_idx, col]
            ax.axis("off")

            if col >= len(class_samples):
                continue

            image_path, _, _ = class_samples[col]

            with Image.open(image_path) as img:
                ax.imshow(img.convert("RGB"))

            ax.set_title(
                f"{split_name.title()} | {class_name}\n{Path(image_path).name}",
                fontsize=9
            )

    fig.suptitle(
        f"Representative Raw Images — {split_name.title()} Test Set",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()


show_representative_samples(test_internal_raw, "internal", n_per_class=3)
show_representative_samples(test_external_raw, "external", n_per_class=3)


### 6.4 Initial inspection notes

Complete this interpretation **after running the cells above**.

Record only observations directly supported by the data.

**Class balance**
- Internal:
- External:
- Material difference observed:

**Native image dimensions**
- Internal:
- External:
- Material difference observed:

**Visual comparison**
- Internal / studio characteristics:
- External / mobile characteristics:
- Candidate distribution shifts worth testing:

### Current audit status
At the end of Section 6 we have characterised the two supplied datasets, but we have **not yet diagnosed the model failure**.

The next stage is **Section 7 — Baseline Evaluation**, where the supplied model's predictions will be used to quantify the internal-versus-external performance gap using accuracy and class-sensitive metrics. Those results will determine which candidate shifts from this inspection are worth investigating experimentally.
